# Instance Segmentation with LTDETRv2

This notebook demonstrates how to use LightlyTrain for instance segmentation with our state-of-the-art LTDETRv2 model and publicly released weights trained on the [COCO](https://arxiv.org/abs/1612.03716) dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/ltdetr_instance_segmentation.ipynb)

> **Important**: When running on Google Colab make sure to select a GPU runtime for faster processing. You can do this by going to `Runtime` > `Change runtime type` and selecting a GPU hardware accelerator.

## Installation

In [ ]:
!pip install lightly-train

> **Important**: LightlyTrain is officially supported on
> - Linux: CPU or CUDA
> - MacOS: CPU only
> - Windows (experimental): CPU or CUDA
>
> We are planning to support MPS for MacOS.
>
> Check the [installation instructions](https://docs.lightly.ai/train/stable/installation.html) for more details on installation.

## Train an instance segmentation model

Training your own LTDETRv2 instance segmentation model is straightforward with LightlyTrain.

### Download dataset

First download a small example dataset in YOLO segmentation format. The archive contains the images, the polygon labels and a `config.yaml` with the dataset paths and class names.

In [ ]:
!wget https://github.com/lightly-ai/coco128_yolo/releases/download/v0.0.3/instance_segmentation.zip && unzip -q instance_segmentation.zip

The dataset looks like this after the download completes:

```
instance_segmentation
├── config.yaml
├── images
│   ├── train2017
│   │   ├── 000000000009.jpg
│   │   ├── ...
│   │   └── 000000000536.jpg
│   └── val2017
│       ├── 000000000139.jpg
│       ├── ...
│       └── 000000009891.jpg
└── labels
    ├── train2017
    │   ├── 000000000009.txt
    │   ├── ...
    │   └── 000000000536.txt
    └── val2017
        ├── 000000000139.txt
        ├── ...
        └── 000000009891.txt
```

Paths in `config.yaml` are resolved relative to the config file itself. `data` also accepts a dictionary with the same keys, see the [documentation](https://docs.lightly.ai/train/stable/instance_segmentation/ltdetrv2.html#data) for details.

Then start training with `train_instance_segmentation`. LightlyTrain automatically configures the model and augmentations, while still allowing every setting to be customized.

In [ ]:
import lightly_train

lightly_train.train_instance_segmentation(
    out="out/my_experiment",
    model="ltdetrv2-seg-s-coco",
    steps=100,  # Small number of steps for demonstration, default is 273_504.
    batch_size=4,  # Small batch size for demonstration, default is 32.
    data="instance_segmentation/config.yaml",
)

Once training completes, the final model checkpoint is saved in `out/my_experiment/exported_models/exported_last.pt`. The best model according to the validation mask mAP is saved in `out/my_experiment/exported_models/exported_best.pt`.

## Predict with your own LTDETRv2 weights

In [ ]:
!wget -O image.jpg http://images.cocodataset.org/val2017/000000039769.jpg

### Load the model weights

Load the best model exported during training with LightlyTrain's `load_model` function:

In [ ]:
model = lightly_train.load_model("out/my_experiment/exported_models/exported_best.pt")

### Predict and visualize the instances

Run prediction on the example image with your trained model and visualize its masks, bounding boxes, and class labels:

In [ ]:
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks

prediction = model.predict("image.jpg")

image = read_image("image.jpg")
image_with_masks = draw_segmentation_masks(image, masks=prediction["masks"], alpha=0.6)
image_with_instances = draw_bounding_boxes(
    image_with_masks,
    boxes=prediction["bboxes"],
    labels=[model.classes[label.item()] for label in prediction["labels"]],
)
plt.imshow(image_with_instances.permute(1, 2, 0))
plt.axis("off")
plt.show()

## Next Steps

- [Instance Segmentation Documentation](https://docs.lightly.ai/train/stable/instance_segmentation/index.html): Learn more about training, inference, data formats, and available models.
- [LTDETRv2 Instance Segmentation Model Export](https://colab.research.google.com/github/lightly-ai/lightly-train/blob/main/examples/notebooks/ltdetr_instance_segmentation_export.ipynb): Export an LTDETRv2 instance segmentation model to ONNX and TensorRT.